# 🎙️ TTT-Dialect: 노인·방언 화자 개인 적응 음성 인식

**3단계 파이프라인**
1. 방언 자동 감지 (Dialect Identifier)
2. 방언별 Whisper 특화 모델
3. TTT 개인 적응 (Test-Time Training)

---
**시작 전 확인:** 런타임 → 런타임 유형 변경 → **T4 GPU** 선택

## Step 1 — Google Drive 연동

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/TTT-Dialect'
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data/raw/dialect', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data/raw/elderly', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data/processed', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)
print('✅ Drive 연동 완료!')
print(f'📁 저장 경로: {DRIVE_ROOT}')

## Step 2 — 저장소 클론 및 패키지 설치

In [ ]:
import os

REPO_DIR = '/content/TTT-Dialect'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kts6450/TTT-Dialect.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
    print('✅ 최신 코드로 업데이트 완료')

%cd {REPO_DIR}
print(f'📁 현재 디렉토리: {os.getcwd()}')

In [ ]:
!pip install -q \
    openai-whisper \
    transformers \
    librosa \
    soundfile \
    jiwer \
    loguru \
    pyyaml \
    accelerate \
    datasets \
    torchaudio
print('✅ 패키지 설치 완료!')

## Step 3 — GPU 확인

In [ ]:
import torch

print(f'GPU 사용 가능: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 모델: {torch.cuda.get_device_name(0)}')
    print(f'GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ GPU가 없습니다. 런타임 → 런타임 유형 변경 → T4 GPU 선택 후 재시작하세요.')

## Step 4 — 베이스 모델 로드

HuggingFace에 공개된 **한국어 파인튜닝 Whisper 모델**을 베이스로 사용합니다.  
직접 파인튜닝하는 시간을 절약하고, TTT 부분에 집중할 수 있습니다.

In [ ]:
import sys
sys.path.insert(0, '/content/TTT-Dialect')

from models.base_whisper import KoreanWhisperModel

# 한국어 파인튜닝 완료된 공개 모델 사용
MODEL_NAME = 'SungBeom/whisper-small-ko'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'모델 로드 중: {MODEL_NAME}')
model = KoreanWhisperModel(MODEL_NAME)
model.model = model.model.to(DEVICE)

params = model.count_parameters()
print(f'\n✅ 모델 로드 완료!')
print(f'   총 파라미터: {params["total"]:,}')
print(f'   디바이스: {DEVICE}')

## Step 5 — KSS 샘플 데이터 다운로드 (AI Hub 데이터 없을 때)

In [ ]:
# AI Hub 승인 전 → Common Voice 한국어 샘플로 파이프라인 검증
from datasets import load_dataset
import soundfile as sf
import numpy as np
import json
from pathlib import Path

SAMPLE_DIR = Path('/content/TTT-Dialect/data/raw/kss_sample')
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

print('Common Voice 한국어 샘플 다운로드 중 (500개)...')
ds = load_dataset(
    'mozilla-foundation/common_voice_13_0',
    'ko',
    split='train[:500]',
    trust_remote_code=True
)

manifest = []
SR = 16000

for i, item in enumerate(ds):
    audio = np.array(item['audio']['array'], dtype=np.float32)
    orig_sr = item['audio']['sampling_rate']
    
    # 16kHz 리샘플링
    if orig_sr != SR:
        import librosa
        audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=SR)
    
    path = SAMPLE_DIR / f'sample_{i:04d}.wav'
    sf.write(str(path), audio, SR)
    
    manifest.append({
        'audio_path': str(path),
        'transcript': item['sentence'],
        'dialect': '서울',
        'speaker_age': 0,
        'speaker_id': item.get('client_id', f'anon_{i}')[:8],
        'duration_sec': len(audio) / SR
    })

manifest_path = SAMPLE_DIR / 'manifest.jsonl'
with open(manifest_path, 'w', encoding='utf-8') as f:
    for m in manifest:
        f.write(json.dumps(m, ensure_ascii=False) + '\n')

print(f'✅ 샘플 {len(manifest)}개 다운로드 완료!')
print(f'   저장 경로: {manifest_path}')

## Step 6 — TTT 적응 어댑터 초기화

In [ ]:
from models.ttt_adapter import TTTAdapter

PROFILE_DIR = '/content/TTT-Dialect/data/user_profiles'

adapter = TTTAdapter(
    base_model=model,
    profile_dir=PROFILE_DIR,
    top_k_layers=2,       # 상위 2개 레이어만 업데이트
    lr=1e-4,
    adaptation_steps=30,
    gradient_clip=1.0,
)

print('✅ TTT 어댑터 초기화 완료!')
print(f'   적응 레이어: 상위 {adapter.top_k_layers}개')
print(f'   학습률: {adapter.lr}')
print(f'   적응 스텝: {adapter.adaptation_steps}')

## Step 7 — TTT 전 WER 베이스라인 측정

In [ ]:
import json
import librosa
import torch
from jiwer import wer
from tqdm.notebook import tqdm

SR = 16000
N_TEST = 50  # 테스트 샘플 수

# manifest 로드
samples = []
with open('/content/TTT-Dialect/data/raw/kss_sample/manifest.jsonl') as f:
    for line in f:
        samples.append(json.loads(line))

test_samples = samples[:N_TEST]

def infer(sample, mdl):
    audio, _ = librosa.load(sample['audio_path'], sr=SR)
    feat = mdl.processor.feature_extractor(
        audio, sampling_rate=SR, return_tensors='pt'
    ).input_features.to(DEVICE)
    with torch.no_grad():
        result = mdl.transcribe(feat)
    return result[0] if result else ''

print('TTT 이전 WER 측정 중...')
refs, hyps = [], []
for s in tqdm(test_samples):
    refs.append(s['transcript'])
    hyps.append(infer(s, model))

wer_before = wer(refs, hyps)
print(f'\n📊 TTT 이전 WER: {wer_before:.1%}')
print(f'   예시 정답:    "{refs[0][:30]}"')
print(f'   예시 인식:    "{hyps[0][:30]}"')

## Step 8 — TTT 캘리브레이션 실행 (핵심)

In [ ]:
N_CALIB = 20  # 캘리브레이션 샘플 수
calib_samples = samples[N_TEST:N_TEST + N_CALIB]

# 캘리브레이션 특징 추출
calib_features = []
calib_texts = []

for s in calib_samples:
    audio, _ = librosa.load(s['audio_path'], sr=SR)
    feat = model.processor.feature_extractor(
        audio, sampling_rate=SR, return_tensors='pt'
    ).input_features[0]
    calib_features.append(feat)
    calib_texts.append(s['transcript'])

print(f'캘리브레이션 시작: {N_CALIB}개 샘플')
print('(약 1~2분 소요)\n')

profile = adapter.calibrate(
    user_id='test_user_colab',
    audio_features=calib_features,
    transcripts=calib_texts,
    dialect='서울',
    age=70,
)

print(f'\n✅ TTT 캘리브레이션 완료!')
print(f'   WER 이전: {profile.wer_before:.1%}')
print(f'   WER 이후: {profile.wer_after:.1%}')
print(f'   개선: {profile.wer_before - profile.wer_after:.1%}p')

## Step 9 — TTT 전후 WER 비교 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('TTT 적용 전·후 성능 비교', fontsize=16, fontweight='bold')

# 1. WER 비교 막대
ax = axes[0]
labels = ['TTT 이전', 'TTT 이후']
values = [profile.wer_before * 100, profile.wer_after * 100]
colors = ['#FF6B6B', '#4ECDC4']
bars = ax.bar(labels, values, color=colors, alpha=0.85, width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=13)
ax.set_ylabel('WER (%) — 낮을수록 좋음')
ax.set_title('WER 비교')
ax.set_ylim(0, max(values) * 1.3)
ax.spines[['top', 'right']].set_visible(False)

# 2. 적응 이력 추이
ax2 = axes[1]
if profile.adaptation_history:
    steps = list(range(len(profile.adaptation_history)))
    wers = [h['wer'] * 100 for h in profile.adaptation_history]
    ax2.plot(steps, wers, 'o-', color='#667eea', linewidth=2.5, markersize=8)
    ax2.fill_between(steps, wers, alpha=0.15, color='#667eea')
    ax2.set_xlabel('적응 횟수')
    ax2.set_ylabel('WER (%)')
    ax2.set_title('적응에 따른 WER 변화')
    ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()

# Drive에 저장
save_path = f'{DRIVE_ROOT}/results/wer_comparison.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n💾 그래프 저장: {save_path}')

## Step 10 — 모델 체크포인트 Drive에 저장

In [ ]:
import shutil

# 사용자 프로파일 & 적응 모델을 Drive에 백업
src_profile = '/content/TTT-Dialect/data/user_profiles'
dst_profile = f'{DRIVE_ROOT}/checkpoints/user_profiles'

if os.path.exists(src_profile):
    shutil.copytree(src_profile, dst_profile, dirs_exist_ok=True)
    print(f'✅ 프로파일 저장: {dst_profile}')

# 결과 요약 저장
import json
summary = {
    'model': MODEL_NAME,
    'n_calibration_samples': N_CALIB,
    'wer_before': profile.wer_before,
    'wer_after': profile.wer_after,
    'improvement_pp': profile.wer_before - profile.wer_after,
    'improvement_pct': (profile.wer_before - profile.wer_after) / profile.wer_before * 100
}

with open(f'{DRIVE_ROOT}/results/summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f'\n📊 최종 결과 요약')
print(f'   TTT 이전 WER : {summary["wer_before"]:.1%}')
print(f'   TTT 이후 WER : {summary["wer_after"]:.1%}')
print(f'   개선량       : {summary["improvement_pp"]:.1%}p')
print(f'   상대 개선율  : {summary["improvement_pct"]:.1f}%')
print(f'\n💾 모든 결과가 Drive에 저장됐습니다!')
print(f'   경로: {DRIVE_ROOT}/results/')

## (선택) AI Hub 데이터 연동

AI Hub 승인 후 Drive에 데이터를 올리고 아래 셀을 실행하세요.

In [ ]:
# AI Hub 데이터가 Drive에 업로드된 후 실행
DIALECT_PATH = f'{DRIVE_ROOT}/data/raw/dialect'
ELDERLY_PATH = f'{DRIVE_ROOT}/data/raw/elderly'
OUTPUT_PATH  = f'{DRIVE_ROOT}/data/processed'

# 방언 데이터 전처리
!python data/preprocess.py \
    --dialect_path {DIALECT_PATH} \
    --elderly_path {ELDERLY_PATH} \
    --output_path  {OUTPUT_PATH}

print('✅ AI Hub 데이터 전처리 완료!')